# VISCERA · Joint shape + colour + disease VAE

Choose **Runtime → Run all**. Approve the Google Drive prompt when Google asks.
No source edits, API keys or manual dataset selection are needed. Keep the three
matching ZIP files beside this notebook in **VISCERA Joint VAE Training 2026-10-07**.
The final cell downloads **viscera-joint-weights.zip**; a copy remains in Drive.

This is an actual conditional VAE with a shared latent representation and separate
shape, material and disease heads. It never substitutes ridge regression. Training
uses paired rendered surface positions, linear RGB and roughness, with a frozen
renderer surrogate connecting observation losses to the generated editor controls.
The decoder retains the full reference mesh and supports texture export in Studio.

**Two separate datasets:** 2,304 baseline examples and 4,800 disease examples;
24 references, 12 organ categories and 25 supported disease profiles. Shared
synthetic families stay in the same train/validation/audit split across both sets.
These are authored examples, not 7,104 independent patients or measured tissue
colour. Additional DRENDS RGB/3D observations are a separately supplied porcine
benchmark, excluded from this human disease training run.

GPU execution is requested in the notebook metadata. CPU execution also works;
runtime allocation and limits are controlled by [Google Colab](https://research.google.com/colaboratory/faq.html).
No language or physics model is retrained in this notebook.


In [ ]:
"""Standard-library-only, checksum-bound joint Colab setup helpers."""

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath


def file_digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def extract_package(archive_path: Path, expected: str, destination: Path, prefix: str) -> Path:
    if not re.fullmatch(r"[a-f0-9]{64}", expected) or file_digest(archive_path) != expected:
        raise ValueError("Archive does not match this notebook: " + archive_path.name)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination / prefix
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        names = [item.filename for item in entries]
        if (
            len(names) != len(set(names))
            or len(entries) > 10000
            or sum(i.file_size for i in entries) > 2_000_000_000
        ):
            raise ValueError("Archive entry count or expanded size is invalid")
        for item in entries:
            parts = item.filename.split("/")
            if (
                len(parts) < 2
                or parts[0] != prefix
                or any(p in ("", ".", "..") for p in parts)
                or "\\" in item.filename
                or ":" in item.filename
                or item.flag_bits & 1
                or item.is_dir()
                or stat.S_IFMT(item.external_attr >> 16) not in (0, stat.S_IFREG)
            ):
                raise ValueError("Unsafe ZIP member")
        manifest_name = prefix + "/package-manifest.json"
        manifest_bytes = archive.read(manifest_name)
        manifest = json.loads(manifest_bytes)
        hashes = manifest.get("artifact_hashes", {})
        if manifest.get("format") != "viscera-joint-package-v1" or not hashes:
            raise ValueError("Missing package manifest")
        if set(names) != {manifest_name} | {prefix + "/" + key for key in hashes}:
            raise ValueError("Package has extra or missing members")

        def verify(target: Path) -> None:
            if target.is_symlink() or any(p.is_symlink() for p in target.rglob("*")):
                raise ValueError("Extraction must not contain symlinks")
            if (target / "package-manifest.json").read_bytes() != manifest_bytes:
                raise ValueError("Cached package manifest differs from the pinned archive")
            actual = {p.relative_to(target).as_posix() for p in target.rglob("*") if p.is_file()}
            if actual != set(hashes) | {"package-manifest.json"}:
                raise ValueError("Extraction has extra or missing files")
            for name, sha in hashes.items():
                path = target / name
                if (
                    path.is_symlink()
                    or not path.resolve().is_relative_to(target.resolve())
                    or file_digest(path) != sha
                ):
                    raise ValueError("Extracted file checksum mismatch: " + name)

        if root.exists():
            verify(root)
            return root
        temporary = Path(tempfile.mkdtemp(prefix=".joint-extract-", dir=destination))
        try:
            for item in entries:
                target = temporary.joinpath(*PurePosixPath(item.filename).parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output, 1024 * 1024)
            verify(temporary / prefix)
            (temporary / prefix).replace(root)
        finally:
            shutil.rmtree(temporary)
    return root


def install_environment(root: Path, environment: Path) -> Path:
    override = os.environ.get("VISCERA_TRAIN_PYTHON")
    # Resolving a venv's interpreter symlink would silently select the base
    # interpreter and install into the wrong environment (PEP 668 on macOS).
    python = Path(override).expanduser().absolute() if override else environment / "bin/python"
    if not override:
        if not python.is_file():
            result = subprocess.run([sys.executable, "-m", "venv", str(environment)])
            if result.returncode:
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "virtualenv==20.35.3"], check=True
                )
                subprocess.run([sys.executable, "-m", "virtualenv", str(environment)], check=True)
        host = subprocess.run(
            [
                sys.executable,
                "-c",
                "import torch,json; print(json.dumps({'version':torch.__version__.split('+')[0],'cuda':torch.version.cuda,'available':torch.cuda.is_available()}))",
            ],
            capture_output=True,
            text=True,
        )
        info = (
            json.loads(host.stdout)
            if host.returncode == 0
            else {"version": "2.11.0", "cuda": None, "available": False}
        )
        if not re.fullmatch(r"2\.\d+\.\d+", info["version"]):
            raise ValueError("Unsupported host PyTorch version")
        variant = (
            "cu" + info["cuda"].replace(".", "") if info["available"] and info["cuda"] else "cpu"
        )
        command = [
            str(python),
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "torch==" + info["version"],
        ]
        if sys.platform == "linux":
            command += ["--index-url", "https://download.pytorch.org/whl/" + variant]
        subprocess.run(command, check=True)
    subprocess.run(
        [
            str(python),
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-r",
            str(root / "notebooks/requirements-joint-colab.txt"),
        ],
        check=True,
    )
    subprocess.run(
        [
            str(python),
            "-c",
            "import numpy,scipy,torch,trimesh,pydantic; print('Joint training dependencies ready; CUDA:', torch.cuda.is_available())",
        ],
        check=True,
    )
    return python


def verify_weights(path: Path, receipt: dict) -> str:
    actual = file_digest(path)
    if receipt.get("status") != "complete" or receipt.get("weights_sha256") != actual:
        raise ValueError("Weight export did not complete or its checksum changed")
    with zipfile.ZipFile(path) as archive:
        entries = archive.infolist()
        names = [entry.filename for entry in entries]
        prefix = "viscera-joint-weights/"
        if len(entries) > 1000 or sum(entry.file_size for entry in entries) > 512 * 1024 * 1024:
            raise ValueError("Joint weight archive exceeds its size limit")
        for entry in entries:
            parts = entry.filename.split("/")
            if (
                len(parts) < 2
                or parts[0] != prefix[:-1]
                or any(p in ("", ".", "..") for p in parts)
                or "\\" in entry.filename
                or ":" in entry.filename
                or entry.flag_bits & 1
                or entry.is_dir()
                or stat.S_IFMT(entry.external_attr >> 16) not in (0, stat.S_IFREG)
            ):
                raise ValueError("Unsafe weight archive member")
        manifest = json.loads(archive.read(prefix + "manifest.json"))
        hashes = manifest.get("artifact_hashes", {})
        required = {
            "model.npz",
            "model.json",
            "training_report.json",
            "exact-renderer-audit.json",
            "data/baseline-release.json",
            "data/disease-release.json",
        }
        if (
            manifest.get("format") != "viscera-joint-vae-v1"
            or manifest.get("smoke_only") is not False
            or manifest.get("profile") != "full"
            or not required <= hashes.keys()
            or len(names) != len(set(names))
            or set(names) != {prefix + "manifest.json"} | {prefix + k for k in hashes}
        ):
            raise ValueError("Invalid joint weight archive")
        for relative, expected in hashes.items():
            if hashlib.sha256(archive.read(prefix + relative)).hexdigest() != expected:
                raise ValueError("Weight member checksum mismatch")
        if hashlib.sha256(archive.read(prefix + "manifest.json")).hexdigest() != receipt.get(
            "export_manifest_sha256"
        ):
            raise ValueError("Export manifest changed after completion")
        audit = json.loads(archive.read(prefix + "exact-renderer-audit.json"))
        if audit.get("passed") is not True:
            raise ValueError("Exact renderer checks did not pass")
    return actual


PACKAGES = {'baseline': {'filename': 'viscera-joint-baseline-2026-10-07.zip', 'prefix': 'baseline', 'sha256': '21178d5226e9daddf8f033055ea97d3d664d04d98c1394723b3217d55ec85c64', 'bytes': 8522938, 'files': 10}, 'disease': {'filename': 'viscera-joint-disease-2026-10-07.zip', 'prefix': 'disease', 'sha256': 'fd53b4879d8ebb15d50b5381ebd1ceb85cba3ce97bfa311f8af20d700da893e7', 'bytes': 18048969, 'files': 10}, 'runtime': {'filename': 'viscera-joint-runtime-2026-10-07.zip', 'prefix': 'viscera', 'sha256': '48fafaa777b14bec4652b3bcc339fcc8a5304aa03fbb9e2d4c71c059e0ffbf2d', 'bytes': 25400977, 'files': 108}}
DRIVE_FOLDER_NAME = 'VISCERA Joint VAE Training 2026-10-07'
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
if os.environ.get("VISCERA_JOINT_ARCHIVES"):
    DRIVE_FOLDER = Path(os.environ["VISCERA_JOINT_ARCHIVES"]).resolve()
elif IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
else:
    DRIVE_FOLDER = Path.cwd()
WORK = Path(os.environ.get("VISCERA_JOINT_WORK", "/content/viscera-joint" if IS_COLAB else str(Path.cwd() / "viscera-joint-work"))).resolve()
WORK.mkdir(parents=True, exist_ok=True)
PATHS = {}
for lane, package in PACKAGES.items():
    source = DRIVE_FOLDER / package["filename"]
    if not source.is_file():
        raise FileNotFoundError(f"Missing {source}. Keep all three release ZIPs in the named Drive folder.")
    local = WORK / package["filename"]
    if not local.exists() or file_digest(local) != package["sha256"]:
        shutil.copyfile(source, local)
    PATHS[lane] = extract_package(local, package["sha256"], WORK / package["sha256"][:12], package["prefix"])
    print("Verified", lane, package["sha256"])
ROOT = PATHS["runtime"]


## Install an isolated training environment

The notebook kernel remains unchanged. Setup retains the host's PyTorch/CUDA wheel
variant, installs the pinned scientific dependencies and verifies imports in a
fresh process. The first setup can take several minutes.


In [ ]:
TRAIN_PYTHON = install_environment(ROOT, WORK / "train-env")
RUNTIME = json.loads(subprocess.check_output([str(TRAIN_PYTHON), "-c", "import torch,sys,json; print(json.dumps({'python':sys.version.split()[0],'torch':torch.__version__,'cuda':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"], text=True))
run_identity = hashlib.sha256(json.dumps({"packages": PACKAGES, "runtime": RUNTIME}, sort_keys=True).encode()).hexdigest()[:16]
OUT = Path(os.environ.get("VISCERA_JOINT_OUT", str(DRIVE_FOLDER / "runs" / run_identity)))
OUT.mkdir(parents=True, exist_ok=True)
print("Runtime:", RUNTIME)
print("Outputs:", OUT)


## Train the joint VAE, evaluate, then verify the actual rendered outputs

The default run trains the observation surrogate for 1,400 steps and the VAE for
4,000 steps. Training uses KL warmup, free bits, a learned conditional prior,
balanced condition sampling and cross-modal reconstruction/distribution losses.
Checkpoint selection uses validation only; audit results are measured after the
checkpoint is frozen. Baseline comparisons and limitations remain in the report.

The export checks seeded replay and actual mesh/colour variation across every
supported condition. It is not a clinical realism or physics certification.
A completed, checksum-verified training stage is reused on rerun; interrupted
training without a completed artifact restarts.


In [ ]:
command = [str(TRAIN_PYTHON), "-u", str(ROOT / "scripts/joint_train.py"),
           "--baseline", str(PATHS["baseline"]), "--disease", str(PATHS["disease"]),
           "--templates", str(ROOT / "templates"), "--out", str(OUT), "--device", "auto"]
environment = dict(os.environ, PYTHONPATH=str(ROOT / "py") + os.pathsep + str(ROOT), PYTHONDONTWRITEBYTECODE="1")
with (OUT / "training.log").open("a") as log:
    process = subprocess.Popen(command, cwd=ROOT, env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = process.wait()
if returncode:
    raise RuntimeError(f"Training/export failed with code {returncode}. See {OUT / 'training.log'}; no successful weights are claimed.")
COMPLETION = json.loads((OUT / "completion.json").read_text())
MODEL = json.loads((OUT / "model/model.json").read_text())
print("Selected method:", MODEL["selected_method"])
print("Validation reconstruction:", json.dumps(MODEL["validation"]["reconstruction"], indent=2))
print("Held-out prior comparisons:", json.dumps(MODEL["audit"]["prior"]["macro"], indent=2))
print("Full renderer audit:", OUT / "exact-renderer-audit.json")


## Download and use in Studio

Open Studio → **Trained models** → choose **viscera-joint-weights.zip** →
**Validate & activate**. Choose an organ and optional disease profile, then use
**Joint VAE variation**. Change the seed for another reproducible result; the
variation temperature controls sampling spread. This action changes mesh,
material/colour and disease controls together.

Existing separate models and text controls remain available. The joint bundle
requires the updated Studio code and matching reference library. Its importer
checks both before activation. The full training report travels inside the ZIP.


In [ ]:
WEIGHTS = OUT / "viscera-joint-weights.zip"
weights_sha = verify_weights(WEIGHTS, COMPLETION)
print("Verified weights:", WEIGHTS)
print("SHA-256:", weights_sha)
if IS_COLAB:
    from google.colab import files
    files.download(str(WEIGHTS))
else:
    print("Local run complete. Import the displayed ZIP in Studio.")
